# Edge-IIoTset Fixed-Test Feature & Dataset Integrity Benchmark

Reproducible, no-test-label tuning. Compare **FedAvgMLP**, **FedAvgKAN**, and the explicitly **centralized** `PooledLightGBM` reference.

- Four leave-one-attack-family-out folds × three seeds.
- **Immutable test and validation samples** across all feature sets and deduplication policies within each fold and seed.
- Three training-only dedup policies: none / full signature / safe30 signature.
- Retained safe30 and expanded36 features; retrain with ports, checksum, or sequence/payload features removed.
- Targets 1% and 5% *validation benign FPR*. Always report **actual test FPR**.
- Two T4 workers if available; ZIP results automatically.

**Important limitations:** Capped class-stratified sample alters attack prevalence; grouped exact-row signatures do not constitute real capture-/device-disjoint testing. LightGBM pools training data centrally, and is not a valid FL communication baseline. Cross-architecture step counts do not imply matched compute cost.

In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess, hashlib
import torch
work = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path('/mnt/data/edge_fixed_notebook_local')
work.mkdir(parents=True, exist_ok=True)
out = work / 'EdgeIIoT_FixedTest_Results'
out.mkdir(parents=True, exist_ok=True)
engine = work / 'edge_fixedtest_engine.py'
print('Python:', sys.version)
print('CUDA devices:', torch.cuda.device_count())
try:
    import lightgbm
    print('LightGBM version:', lightgbm.__version__)
except ImportError:
    print('LightGBM not installed; installing required dependency')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm'], check=True)

In [ ]:
import base64, zlib
encoded_engine = "eNrVfXtz20ay7//6FEhSuwBsCCJlO+slg62j2HKiil9XcvZkLy8LBREgiQgEEACUyPh4P/vtX/cMMCApxd6z51bdVMUCB/Ps6enpN77++uvzeJEcX1wUH+qksebpJomPm6SmxyRq1lViRXlsxUm8LrN0FjVpkVvXST5brqLqxj+6TOokqmZLK06jRV7UTTobWXW6WmdRk8TWq9eetaiKdUnPNXXQ1J41i0r+Ga2oIF94Vl5YsyxKV7UVXRfrhgYrs2K7SvLGKqvkNspouMQ/Os8XWVovj4s823pW3dC8oqzIE8/6KVosssQqKisrZlHmW28La5nGcZJbWMkxFtascxrMP/r666+P0lVZVI1V1EdzahPeJFsrzS3Hfvf+/O33r8+uwrc/vwk//Hh5fvbyyvbsNz+93imhX+e/vL/cKX335n2vxB0d0Rh+kt+mVZFPMM40sIe2Hn8erbNmScvIkurI/EFNousscaIsC5tllURxHXyo1omrW9bb2mvSVeL9Whe593taztMs8ZoqmiXX0ezGq6ifYuVF1aKMqjrxqsRbRvUyS6+9xcyb1bde2iRVUxQZgaAqVlYZNXhrqf7f0095MSuyLJlh12v98kWxzqm1nku+XpVbK6qtvNRFJbamRlkZ67KmICyRPvlR95bnRqGf57p8vs552ChDP6+kUn2TEbblPqFFWRWzpK5pT3WLy+J6XTdXhAAAp1mdKqybRNe7or9ZcsFlOxVXSVOls3alNEQYrWdhPSsIgtFtUkWLJKTBZ2lNU1PlqDVbV7fJTl9FnGRhnSjw6T5/wGm4Wq7n8yy5wok4Orp89+5DAJA7hC2LpCGEcezzlz+ch2c/v7z4EOI9odfJDaP5yV1R3dCybSud80Y5e29cP63DOK0c10qyOrHsk1XenMRRE50kdNjDlPZvUaXNNuTzYrvu0bufPwQY58QGOQA1CC/aWnTICTlre2xRNX91g64JseiACl56ySatm7C4UVj64uzFj+equ50BZ9FsmYS3z/zyJrOPXrz6IYjTWePUSUI4Phl4T0+9J08Hf5l682iVZmlChfZ7gltI+5rjBBMgXr4srsKfX76nx6v/9Zo6/1VATL8vo7wuVnc0N3vqHVl5OKM+MM1ntE3rPMZD3SRlHTz3rqNmtgzr9PckGJ4+97IqGPiDwfCZJ6QjOH3qJavrJI5pVKpB9esE7Y+sVbShvY/ysCru6uAtaFC0jtMmZJoGDCmuo2uafrOlPofferPlOr+RoZ4PBgOvpJNHoCjDMqlCXuk2eIYXRFnTRY5XwfApCo4sEIB6WWRx2NB5ThqAyR8MPfrn2dSjrUhnyyxpwigrlxENhwkq0k3Yh+p2Hc2TJwMCT7LB0UzikA5tQg0JSEzZw7Ig4i7Qzmk5VHW+JuJT02S4JyqQTowiAJiRHK1eJfHZ7eLNa2yKPP909paq4G5Ib5PgAbTyiX7ZRNrm1mH0v3rz7qdz2w1AOomiEs7465KQOWmx5kFs6aPI1Otw4onGiaHCiVNakfHfPbtkbtJf6OeXYMPQHzy00vNzdW3QImnbsDx7Gkzo+Dgb18JltcFNZVwq+82nPl+0Di3dnR4dxcnciolEO6W3cUdML0rXv6MNSMIm2TQOrhAfNWpn46WEHXlDgKBmuJCCLFpdx5F1O7JufWqyIpJC0yfyl+MCniXOrZeXtJYc6KTITd1Uzq3renRtF3x47HUzf05bjLlgUXSvOTWWKfeUjzIqGFNPOyVyK6yifB2BlpqFhNtAY31BOqs0d049As2sXIcz3FE0V4LYKVE3zFlazdZxBOoY3UZpxpesOzLeGCOFMkuedF5UK9qBUZUQ5ueWbfu/FjTcjLdkhi3BmjeunxV3SSUwmtEwUUYzdGgCV2evzgk3o6r0i5KgggOFH8s7H1SBfqWzVUlr+i3M2l/rfF0nMf1aNk3pzwo6OHlD7/NFs9SlTVaHuFfodzMrfbr81RNVzwXn/Xma7xdWdbNfWG/zg4Vdt/MsWtTmszEkzYye1jHNiniTkChDE1FBnNf+b9XW/21t/Gi2ZaJ+ElBp1+tVWtdCxHuFTVglv60Tnu3hFyEvb/Vb02DWmJQ/oys4J2pd77xofy/jqvdbps6Pq3qhJsc/6fQ2RWi8b4oynZkFxBbg8boDgTwTw0lzi+3p0TfWe6LhJbEN85Q432Qzy9ZEd62LlyeEg0kGNCVCTvC2oqYhgFpgrpOKuPEki+uRFd0WaWytmduRW4A442JDBM8/+v7sJTCL29HY3CH9JY6al08se7Ioqi0oeDUjSktgJC4df1NCuGomDzEDWErCZaF/UbH+VRfrapZIBwT1NGdZgH9ju4kcrPA8r6JVwgjQ/qAzcM0wmtPx8NNYPfHDmv9dRTM1FTzJXJJmqcrwJGWM8uB1QzAzukDhgc9X1rpKd8uNIrUWfqbDVYXE0OVNd8SKmzQxsDSPVklv3w0k2R2FeMdlEYdVdNe9qUvimxO/XFbEOxAiCDkW0aWKQ2LEwyURrwTsVLMERSTx5JJK6NZPwHuvV7klNay7lPon6SjNbwtweYrLti1sxG1ivbj6u8XcfuVDxuH6VlEmOfft2bQBPYJ8/PyY7nIqrKqiqgObWOqM5Afby5M7ksvoyrZdsN5E2ZJoRVOzKp5IQLP25dGRd3R7EvFvglcRUX+XKzKZzHG9SE1XrZx4CF41M0S1zEyWWRtMEt3Ld2GW0hnn61TB5YoHs4SjptMjh0Odivdbgk5u1y1wj0mcqaJqy3CRSfhH1psoI5K9IhGX0I/ggyvbIm6RuPaUpVIIu3x5JPEYgqkSZV6clNuIAHVngJluQQsE2ufZHVmAC8+J1yDzd4aDkNg4/E+A0VMP6O5y1LKpeFEUcTAYX0f4lwcPBv+W/fOuScygixn86+D06SP886/uqWBh0NtTi+VtFg6JSeHLMPX4OkyYwQSPJu3kRuSLUsFgSq1xvKilVJmkU+kBtdp+UY2aEl11uLb7VYBn3QtdyVFKLMffo2ydnAMSztx+k4pciK3SFPSjbgFlgOrqk40lsBgQTDAQhsce0wRkiYCQoMPjYIhnmkqLm9j/vGgsoGiHN39r34+uqZMb1QqTpjdq+goo3D1NIFa9oyL9+i54NiorMH02LSF8c/b61bvLN+cvw/+8ePnhR9vjcYiyCwLZnu6b+XtepJSpUbx5tq6XSjSzeEE5ke91cmSp1fvQyOSxM6Fe7tkGbglE7QCBIbi5+7egO7uypi3ATtK//5Lo9CvcAlJVH/ZAdkCtv90AXj/9Gn12ewNOULqcX4Yv3r15//r8w7l1G2VpLEy57WHquF3U8dfFBG06WdEd/wbblwcavibUmHgxMMAvMjv5WbKKVSopmHgoougNMQ6dKFyvipskhDbLp+Nny/KBUKXP8nTtCIJUdHw7zlgx5iGVOn8ZuuNrmtZNLeATdjSrvdyrl+kcd4U1cey3tGSS8r3hE5JWBq7n7MpIz04H3inKO8n66fOBN/SHKNyRsZ8+GXj+k2d4Y0jbJLhTMXfyPTEicVEQvXr6V+rEnQpKbAKasp/zXBwWhZ3cGz5z3fFmMvIG08cBz5l/DfWvk1NuGwU9VNi0aAC+ejIaPpu642iyc+symz0NHAzLUE+q2jklEDwbPPNy1/WjGoyeQ7xI1HD7M2aiQmb/pgEB0igUvmoaAN3ozVeBBqtCYt4GfZAiLowxazpshDaOvPdIbiwq6EPiZCO4NTZrTWIv9kVs5JnKnod0pzVJ8MSdHmiPgb6xzjcRMY8kuh8LQ6l1t8z3Eh0n6YfYzoQYQCKWUqV9FRMaz0gG8nnOJMLMJjjasTsNYj/Fz+GUJlhuHZqs8fp46O1AbAdV7qutQTmUAYm9Jvwn+VRWtctFlF53eiw+MxYzfyQ9FgXUG1pz1heJLy7efQiJLNhTd3rovLbvldCKA8Qd0uFrmBj2dWtpXq4b2/XkPGMGdJHYoIt8a/CdgeZAdjpYcQoNBc2u4tEryJxgXLVKLqMD7lR+tciKa8d+xBTAdduqhvKuvZlKvj7p5JIU3I3g3SRbLaVHI2eALoh/zY+heEvTorHRLGJWthVPueMhXxIQXCOX/uNT2lRbOa3L4KMSeju1w0G+1f2kby4lgzAuoPqSL0VVKnvOxaNuY/ndMolPRA+Jnkg4SsqGEBp/CItGJckxR3piSokqe7JcXwNNRHXaFvlqb8K4uMvpaMeOvSqWtPo4WhFbRPxQFS1ONHSo4vFsS+IJ8QWzNejysWp/XMyPqcIxwxA6hL1N2N2+/6+3ooWfFa2Jla4sjeIPbMpuMVjLZDPSl7KYZX6kLudRlsEwYa3zVvEyssFjOsnGnYyI1k37bIqwda/oxLwtmldQ0wl3JzRkqSfZs129eX3y8u1bZvuoPjHoyi50FpOojeMLOVtMKxa0kFWeNCftJH1b3fNzokMhXTA1M1DYDJqvML6bbgfwTi5s3hq68Q2Y07VM8nG1/exWsie0biLh/BxMDrTaMHsAUxXV8WQIl7cW5VG+da7xdtMiF3rgIq0gCJVqI+pRYpdIDESz3qB6JsZ0USy4eCN936AEd/Cn6ZFmCdGR+93zfcbcvshrkkegZ84b64qqHT8ZaCmO+untZU1kYRUBIgpFuxV7emYehpI9U/rsUJo5coX2K2Ijtf6b2KWx1sIw69Tq8XQTHL084LXOXGHNAF8uuGZA5B10vz976Y5atY5mAmbu2OC0a1zzdNWpmaopTmbTVoCbFXQ/J7YarSZWqckjx/VXSZQ7xGP7zwe81/Rmnae/rRMnrooyj+TI/C04HenldTOgvkBgRvtzM4DBdxHMMD4MWETEat7Ex7qC2+2CLvJ0hwJ+2OOIC2SGVR3+95fn788uz8Pvz3+4eDuyoJs5xgVosWLqmI4+SRasDhOBEwLbihWjPTqA6/YzLE6WaKe8okoXaR5oVn2MkxMcIpZS3z26F6+CHTJwJAbrRI70riJDDY+axNOCc2s1GPjpGuBuQRd8DtLuwlMtVGinWoTYZgIR+rhVZ+lRpe1W0tHPMlWoR0IHsF/IfG2PjQ89q4Y9pXb32TRUg/ve2zuU/RvrDDWt1pBUW7g5lC3AqtfXOP/RrCrqmtVgQB2FJGPgYUQnrqQLmxGJjTRmbUKRFPocBo2P4b5GB19bMIDTWMltUm0NTlkqKl3b2Poa8Gsrv3v7+h+atrQU6xYkreauf6RqbCJne7BokkiqIC60WKXE0WRbuomy9Cahh+t1w2R6zmIDFYCXqeuUbiPqi3WXkILHjFr8JCanMIWNa6puFP0LMiu2J6ctHhwxFIKPn8YPCIvDp8O/EPjXdXLowGtceDzpDsTUVdztr6Kb6yt27jkC1L/gQ6cSIAzojsMh5BKOhxuIJkGJG7oQc5oG3aOW3AjSrk//4P+0dORZ6cEc26J7LrS9igQ/Q6qQXtQCd2hy790humxOEaJNr4EPq5gzIZGXLtRuBVB8cYNkVTbbkXElGHv4mNVCXA9t6Iywxn9FgK7SKEt/B3mMLOJRizz2rOV6kcAFBqo9IaIYziroksoyhcvAUI2iOBistIz1S2VvyLZjK9o9jozFJRsYMAHr26fH11RDzgR6XfJltm7SzEdhKErSsLiG5KegqBFq2hPrFLQJNDF2MLDXRNy+fQqdS7ltBdo6V5sNHJvizsq2Tm+r9jan3fYJYX+az71j+TuFeTKPclZhMpl8YOY1bGVZRrfu8V/5P3/g3jN9PXG1WwzCtN4jY7PilvWvJmliXx3h2OjcwqaXMj2AzjJLNFmrknm6AbCFrjbRTcIKFOnf6dDF/e4PyK/iXtpufDAyImMI1dFMwbKedLWmrqJE3Vv62avAPbQ0SsvN3eno6np8apuCCQ/LUlZHzh5sqM+WOvEM9ue22+vrG+s9XQcyE1yXBcQ4uinqpKJTJJYTgjW7tsH7LJrRe996WSSiwI2TWRon4uNmsSfEFoBXPcGr4DDgUSdNZolCVn3H9khoqznhmhM71H4V9jRY1m0pg1qX0nP7Qi+HXnTzUbKorNhbgC5zfZ9d7a63LB5ASFVoK2rsqBTi27kv2KybAHux6IgYNmAwJdFkILIptzngCsGIRb02pv4K9SbyfuotDumrMKLaKkwbtzcPszhSHhe6eUAd+nmNuzKhnabBPQMcOPAQ8rlnZoRb6kEDOL8+Hrp/ekZr4KUrFurqxdlbtvjQpdCRXk9uKiqkRp6sVCrV65WzikpsOXvs+HLzE9IRx6Q9ToSVwutdRXs7j+EAE1nMfOVMx9dCDBUcBvZ4645a5S/1NFLi7+U6hyFViU9vC1FpK0a0ZhuFxT5nJPL61otlQhI2sxfNiTYV+f8nB42CMG/sU281BzbJB0C1hSrooaHuDP/QHkCX0Vb8SHg3shV6qBMwstVJF1nzE7Vn/XotPUhlmYt4bdCMQGbZLWyP/Q1fnb25eP2P8MW7n99+uFKq+roP+JWYf4LJXE4J8IyRuN0yRntpCqWgM/cG7nenz0SGVc1Hh7bg5dmHs6vzDzT8388vz344D88vL99djnQb6BdSU8hdkmwLm62y6usJjCz7Mfh31cx9bI/VphK/zm9kcspzBQBwGFomKAmniOFh95LvhoPBF823N0mhBySfCFrZHdB/PLv6McSpaY0qI2Jj0wyIRfBblVA6t4yDXDH9s7XD+V81hGiWUsYfQ0Efa2LAa4IeoG6A1gU2gvhqcNX0LOZdxfGAB4EETC3FJJrE9Ai2V8nF9BTVUVVFW6e7nzzhOMAzeiLNEsbT7Uz8cKtzugNt69saxD0xZa5aTg/hPLG/cnW6IivuvJOL0/WUz1g3R32NMW/SDL91Pa0n6mbcXYxqxlgXbj0X6ACyYazJqCwT8mRsAy5HVp/y8C7VrdaDlkYvPaXqYXsRm5V3HQBh2UHfhzwBxcSjkIYNYRdvfwivLn54e/bh58tzOqci0fbp4yJQAG+vLTUX4+byo8XCyQNHEREMDoGC4RbKbpsvlWLE1qDvqmii1FVhfp7AEiwmCz//23AKbc2cf3Wdo5iatm963dJLzZ+1lh+NLgKZQNbNVbTRka21snDXa4oG7mYsOslr8/TcK3L/kcAt8+t2SEZduJ724k9CYrqqSIZk+kIwOB5qgmLU470RyiwyCoFMo22obUhSq1YV8vl+hXYglPi5HkfB81A/Cuz7ddquVI2uN0VDw/YO545AOfnsCMC6pehdkFOlfu0el2+sN60QlrS6ALpi6Q+rLmbEOa6Zx1daMM8ikl+Do7wuiPGEU3ZSWcpfXSkORLxTziUtsTkk6bCuUfqge4NooFJqK4r4xB8+QYfNkqQPmtOCCCvcWJRX0HGW3NLa6Chenf/wd+a4eEwfnusynI/lK9ccNoU6fBkz06htEFQSmO/YLC0aQSpg1oUoWj1Pc8Jz1FF2DWljk/zFgpKyFzidUtj7QBROnt4RLYZbGP9Slm8Zot+L1pq1Gi4amkVrx2l3u6/zcjtCyl09Oe30GkXW12q0jTAB4ZCYUZlQzemu9Cpio8iurYJhd36TkffrlMcmTg1hF45jwloU6beYhQy3P1uvQ1lV5QB7qagvIlMuL16Eb84+XF78Qvf265/fvL1SvO3JrjKwR5B5VukmaAUZX5jrudvxUSyQd3ugL7+oZrXGYl2s1S12AAqq/2nHX+xMFg4c/7C93aZ+vYzKxDPHVEW2IFyrWiVmOFJeCgJdEyt3u9WKdXeXSXkvkiPiiaBMJvxWdEX5BhFMGCBQtcD5uCc5sktmj0M+LKV8Y/2UJGUrleoRQLnG3LP0yUxWldZggyAPW6zkABlI4agNHaPWYCqpgEYvoy3snhJZEc95IsKn4B+vPTitRrq/ncYPb/+w7RTIHRXIJejJVAJTN6008vKn719vtSaTQD+YzGPvKsx36fmRJWSyTGc3WeKoNXsceiIvOx5OZudqT4N3P384sWG5pau3u+i4EhtydxwR2Hue22jLMtHyW+L0EJUG73nbY0j3V3oAFPetrXeR8zKNm8o7cJnJUVI+HnV/V/W90b3eMzyEWl5roc7boi5AuWs/W0LzZtEtjQQpUekJlaO60heKpmtWbcumWFRRuaTbk+BXzOux0ux3isdVBKUAz4loP11R8bHymqYbN2ki5fR7pRBewGfVcMDVSjRlCeCeSSrW2k1WzX2fVHlBL1J1VnpGBpimWic+0YGyth52K7XvVossFvNtNfPC3cEleT7KF1AqYT0IXBOxrwtiZP5Vi0vM6yi+4RaWLYhgs+YYVAZqwywqx0xuwFgCoHFySzzHCfzaYWzXEGGwYa7XtAJtXZepEaW4ZloWa4+g2ooLrrxYR1VEG8WEgxZURRAIZfM2Tec4DhJzok6XhVed+zqrHlqOCJFA8CZjq4EpC9ieog49YCkblOBz3/SvfBJkTGVgZK1cKFpVommeFOCWFqEhMDRruBrlfRDsijEqIK5TuDE3lt1F21ovjGFQRUDvotqqTRH7pfKqMnXlgkJnr1+39H+ELrGjjFh5cSykHOFj8Mu1VutaLEFCyztCDgtRnMJjl0q0SJzE6I69j9hKTjNL8xMO0BVm2WIhLpMms2VR1LgfCBUqOozsAb3gMCdquW6WytQUxPNW6jIgJ/JWG0x1SGzSEqvT1+v0BSt0CHdm/BUuYsICs6+7/tvQ/S8pkV7oN7EVHLZBU5v8k/43VaViWlG9GsYVc6M5bm3U+Y1xZ0KZVR35Q8L/io5XLOR07h6z0ILKRMv6ckhoVtWjw2FbIpoDbrUjvf67QAjwqXE0BPXPHhB14T4c+d/JP+WPTE4BkoGIg4b4O7UKMCfdbVg7ctQC3dC7IXaFTlOKuCG33QAFa+npc4EttT8P2o/Fy5h41TyM5sQ9B0YPOqoA8W7hvMhiUAbo3DwEcIE4QJ1DqK7usJZ9d8dbFJvg3+XtRUXz7VOYTPObvLjLhSOPGkK035OqcLDJQYDh3D87hIFD1x2LLm235ha6uvHhTrjhn9HXV9KX23rSqHGhmuTYdkCC+2eFny5SdToVYM8XvudzoxXHAFXwEaN9ahf30Rzxk1IKSqka9JNl1FT1bOGY19UCt6ciltZhYmkR0w7Z9Hqr9aLaZsNeLXKpa9w26FRLB4z9O1LmHRH/lNoP9SZqCVN3TOhEpDJVBvFwIsvw5DU60K/bp8k/WWBQpGbSlk89PZrb+Tm1b93vnhyC/Q7oZXlayWoxLmvgazh0oCoBUQHuXsB4xy90/EgP4sI61RZfTWx8xJ3hifWA/fk9rfTFPaIUAIu6dojQQKvhoTiEMOVxmCZOWLA3DycPJaVDMJQG7NTdNe05L7Ovf+QpPzpqp6JU9YhK8cNwV2XESIwVfVElk4ityfx4DZ+1akYjh9cB5t7t4tbYOTp5U2/gnz5hkvB4OARpFQNYU+lIXqg30AX6G/jD56ou1AS8P6gqD7rmfYMNp2YHT1w4rvbQr+uqqabj26j/krrvBpqOm0S9xiI9jdlHVlQTNgoPAXaIgUbduezmXtXifu20r24j5sm+sFWT3NsKHX52K6gHclbEYzRFMFunReMt9aretvcKAes2oi2Wm0VRDR2K3RL6AP94gGgE3y86m9DxMwwVvivdCQqBn1InwU9N/kzqB3FI71ZXO7x2PR4hVM6TQPKg1TOwMmHLAHXlODxYDWDCOEr1K6RRz8ZQ3IZMCWLWd7LP1j4p05eqJkfcr8y0z/yG5pbJ0gZeB6TPqE0XrrgnXH7/yslz/00RrzNWFYKChCE0LWGI6Ky5l6deXkjSBKEhxKZUJDm2tdwx6vlVskhrGimUYDXHXlSICZWgbGKY6xJuHcen/jMP/0uHqm2NZB8BTeR9BEEfyjVpR7xg7eSprkcC56+o9pr476iiF4+4G5qgK1OfF9VdVMUy8y7Qu22suiVR2jl2nA3dODXdOMnvNLGhe8zVMG33xPGfP+6m5kfXEJZ7tV330aNTl5kAEpMdKlAwfZs00PP+EVxj74ZY1YcgGgXYnljcsCSZhT2VX7xsEtFwiVE3Ig11gNlpw7446PGaezzQX5sg4zMHeLgLNX8648FOE3Oc4WfsGXWhtqyJ8qUji3BeEbJla/kVORu42ncboZhJPjjOL96WDjtTF48lW9i5PRHAWejUWQzAZ45v6FIK2mo+ies1fE4cGzkwEOHziygsJzAYYa9Sukb0dtN+UnNaOt+TH29Gt8hgQNs6Q3Ij5cR947GGWLX1ua5SwHBaBsKyT0cPOP+pC+2JOxbCwPEWptWxQTjn9Xp2wxlDJhIDGHJUC3Qasgtt6gxxGVfRZmw1hNcbB2hsAmOECQhi+4vIO1VvPWLgVFMLP+GkG/amqYprcbZp05o4Ex55J82JPZ0+2puRdsepsS7FXWzg/jVbr2AgQr8rAkgK58woY3JK43o8qutORsfDqQ7zUPDu7ALSsztSIJrcTLXb0q3piwTtcdFwIP8igi3bApevfLrUVFmKT4lZlmAmUFawuy3wTVuwsgUQKTac+VVNMECctmQA3KmaAIZ/n0jCXNR0oA0qOgnyfxrv7KYkPyHAYd/uknSxZLPwWOd00XFHszTWXFcfJmoeYqtRXLGqCPch09XRWu3h+8qHeic0UJkf8aIpHHXUfDmObHAqSlqiUPeSluqfxdHqP52VX2raD1+WrBLjaFYJRlhlAWOh8IzblrUEpzbOk8XBV4PpuHroHClwPno2GDwm4DxiPk9g1ezAmDdIgbiFEU1JGCC+0JOFemtFQeXPlgXNAlU0PdVJiuzpycmpp9wcVezgddeC+tlvwZxB5O60ggux8BHEkLMXlwQJoLztTzPme332O5PQzo3aGGgElYj2y4S6m7rGXo632/1qW65m+EVoC53ZkgehTfdxmMJFFYHoNyHEQSKRalUZCTvBKy3Ts+wT0nGriNiEoMZEgxYkrDgrZ0OHfruVFj7CkvgacccYARuGexT7pkKd5TxoAz8T6DhBEBJoNFNq4e4OUuzVYVrtjtWB092aZ4fXu5K+26NpEmvVVhEHFb8qhSeB/BWTOO5vfa2g4Jbo1iOZ7Z0yPHoc7v57WjqyTk/au0aAD3dB18scxrC9c8yl95xlXJSzQ3TpmCkWTFVVgImVSBKSZAp2HGQn/ZqHu5ML+J0kbfKoBv5/9PTR6aMeXXvEugtFo+gqnx0d/YfgX5qzjpX4bqzZcXUMC8985f3ipfHGuOx7BAmqfCAIDOidFw3TZewQ9idUlw91soo2ztAD8457KEnVrRNv3JPTwdPnOthPR9O7I+lXY4XKdJQuVkUaE+YeOGXXvTPmapRUehK4w0mJFqV3XZV4POJ9jo6+sV5x8scP0C13KR6tc46YgLIuXawryQCJA4VYRksLZR6zRC6rlhBjwFYV6rLTbqer1brh2DvceCed3KF1Iwr9fetlL9lkNGs4QuTD5dnFW47I8JFQpyiyJH4NVP3h+zdi35nhuLOfRuxxhMc8ifmmiglv1I7TC0SAaN06kWHr1WtLcrf4R68ufmlT0gmT109KpxVX/72sdDLa/VnTPLu/Om70r2ZoA0Pe2QIP5IJ78i3X47xKEG6vM53Rgwbi3FKwc9HjDP6ldFjlF1JVKRMNt5e0ROG8rPoJ6o6svcR7wJHQELBFRIaOht+YEim/GD5XOfdkhmwL/XaA7HjVOg/Bchvp5RiHgcJt1sKjgxnXaKvPX+7ma5D9308v98C276aX+xc2d39rD+zk5+3jg7tIDBEt8P95tkn2cOB9brNN0jSQcFLN5kTATpRbbac9VTU+I/IPbTndpO7M3hmUM05KuskjJEoAZRElapeRS8OVU/5g5sdxlcKCJ6F5HCHANldJVboUGgaFmH90/gtRJgKzSr+mEkBxDrXuwOBnXDdGpjZ9duQXnSb9VM241vSLAlu//vrrXooxbZWVZqO9ZbVZSSV7UiRpjSTmhJMZtamK7gsOnWCj+O7jB8TvAhDsC6YK2hAgCX/U4bZ7DXuRxihTUc5GqqBOtueSnfBT3uzQDEIV7w6OHvUIj8QtBOLZZUKXk3X7zK67kFRvPyYV4OjiUsGrd+EkOnExO/J9WCatA47gSsrBO8fdXtDKijuPO7nTynk2lhjB8CoM2peAV43LY3VI6M+4RXK1lENxdJwtq/P/aQECl3uS6Hgr1ctJSzdYTtKJgw5td9uC3X+nj80g8XbP+RdGmSoDFp6/CnS/Iw7TeVWwiUc8nhS0On/vogTir/MUSaTbDCbspyq4yoePLZAcxcHBURqAvdB1PWp/YlS9DQ04vASFd7q2cmJ4dc5ezOEVwf3NWfj23Ydzq59oSf9CdO2sWSPXjnKVJcbTVoPannrYcYPuwMsQCYlN4KtlGtjvtAOa2smqS36mD3iXCETAg2hj9iCBkShN7nzrrMMx7QKCXDQq1xkJRQt6XBJ35LMvxIsCTvENkwtO34fUMbAQ64lyqmcZnWbG1JXJJtE9PjYpAlN+W6cw9XWuq51bGhj0O+TCttkrMhJpmeViXSGeE1J6e3bYDlJ0MWs3X8Ym6grgvMcnTB8a9kspbgPOeIo3tePI6brftcvltGiSKdVVHUxsITh8vXB+SNpp7NdFy+IyG8xigDWjDrBj7ClMUlAbFGIaPxOkmxM/ETjlplyfXhLs+UIfG8Y7bsH3j7CyRMKaRBlWuQtspMxToRThtaTnYZ6Bc89wJ4m+DOt0Y0VxnKqE2x9evDeuxPYkZoSN4mLOOYMqpFNRXkCtq6Qcbowgjl88T4ntMKbV+ruHnX/TLOmhvI7LjkoM3jo9XbzsfLNaFyaCXetMxbKHYYbGWWj5qG4KjDudY5Z56sCfWd1dcCL56sfEtkf1GnBuE4DrDHCHE9bbynXwj1AMCshbd6eyiFuqgvBHnVuV8K46JEq5zR44QbbEsRm0SJMNvbv2fpqzVXSj2Cclx7L76L5XheEbaIZjmzb6bbAXtXTYzeL5F3lZaAvuQTcLZbL9N/lZnO75WZzeE2p1kRNMOZpfDr/9WJkyETvEHuQ9h6Z9MKw1mfvGOtsJHLNNjDa8BMSTjIlAAr2K0AoViNYRDKK9S0lLuTC8JeiX4SyhQSp/OkcI1JIykYLQCXT/AmT+t19ZOuzVZe8CnM9WqSj+Aa1vQdrAseczHQz2fQuqPecC7vCwhwHm2DkYdO5i97gYgGMUb3DtrQTw7jqusgpIPCqoWg17JrvjKe9Aursbpq0S/PGOBIjqDigU6VagxuIKonxGFZsIu7Aw6DtbPyvWGXsGysa3boPqKxvif3i3pH8SpEsSAwSRK4bCsdxMbODN4Tbr//GG9hFHoYRr+MPs+SHtng/DG4brKuTWvjAMKp2UY98RRukL4KJh2uYFr5Q9XMuRBxUMJB2LFeyvpgsIHGNMLxCjOx6r1+WOZkL3ePpXw1EEHRq+Itwf//6srp78lV1G4Mew4zXS9cvP034oIROOWzHp3edOcqAFg7A/EJW0niYHWnDgg8fqTseRWSHcx1PPaE4/MDA/au2m9CDuBMOYUYgOGmPS1OUU6AeR5hVu8A4ZiA29iRaJylhIbJ/Q8J6HCWCjnUwgY5r+JZl+c3CwH/t8GQZjFpZ4sDkRFFZ1tNjeHQdl4qSz8N2pvivMtxgW7w6OqVPVKm5HGCUj6xVrIgUfWmAHGrrsa3WfD4vCfEXWuzstlH0xjN2cglS+ALGVy16KNeHsXK333W1bPx2jgbLvis+O9ukceAb91GF+A5FM/xu+200sLAZnLejNYqrcdOugiVsX3UPRpQ+76lqf46urlybHve7569YH3HXrnrfuN9YFZzUkVnYkkORPHCm+nmSrJMWFYUoAOUyN4KyJokvQAbJQ3ySwkErSzX828aRz+NUT5JsbtfDPvS6/+/6+R7Drcxte2gO+stoMvHG/e9qyTt0WpZtpnx1Ux4QPCKfE2fOifEvy99XF96/PQ7YAhO/fvb548Q/irwRNHttWroLm1cCGnSjd9LFQe1H1MEX8p9DwEI7qJi0I9dnR31gRTr1Vk+CcdFqJe1UunFxl57XoV9wxqzxoG7qsdK2Ogn+13dNuJg9kb6Ne3B1SIlrjfsTQk2+DBC5jWk/cGzmRUXXewZ6Ksq/WdKeqE632PNDPV0FfMaqbGMaEPx69VZX2ta1aoepO2zyOTUgLDLVhT6sDcP9qEhdvPC0KKc29+Px0xr/P/pCU9YVfqFI5OUyJbfxFAlMSHECsMewE3YvdcD2q0H0dCK4Rk0RFeSJ0QQAvIbEaLIRjm8MhnhhpMvKM3qaHAm03ip1UYZgbl9U+JP8eASxBD4gOi93JYhvYK9q0iGRf0KGQ43s764dSnVFLnzbZ2Uza3aTTswlQzh8JmUvaSpLBZoEJe+e3NRFbfEaClU+BMxx4fx3AWXB2sEPYTLO0dOi12a93PDz1hqfuAe+FI6HWAXBtYtOzDW9eurKlAI+2SjfCWBcEu2YhI9tthrLF9Qqat2xxzarUF52ts5X6OE6Ng6uQvwSmzVbB8/bd2+NXr1U4aaSSPbPBP6Ae/dc04guICNB4VCR34QMfK9Qj2evJAGGgIeH1LcF++Iw/eBTTZbAMnnmM7SCX2LfAHzzjWMIVAW+2TCECir87mlXJIpTsuMHQ90hovS5qJDM4HhKz8is8v4Y7PuNQN+h57m2MtzV3CYLkrfjg+erAS04EasNbMXU5r/iY9uBwLeyJqsRuE7BV0trHyJvGtwD8FBo+UgoW9rTz+ZE+r4uCXVn5e0rsXJA5Lm13lcCddl4gOG5Mm7BCzj5pE7KHQx3M7Y8Y6RP4TmQCikmsdNhDOuFYwtYRQiIUXWJ+2GlHFOtRWsGOzY5RhBOVzF/Ln29ev2cJ86ezt+Pdjuh6KDmdsVUnjBsJi4PWKjYcLGoPk/ZCYUKdDXtHagIqTpJCPDuHHdqN1qOCutqIo7GugG3ov2ZeVr/tQQZXO+bQXWX1rVeraCON+6FsGO8Tnlyv8xwhZK1nVcopigOza14VcSDiaRBeb5GxW5JUrFYtjxzVNyGce9mwxHiqvT4gS2DztWSnzcQ2u0JydpUk6dVQn/QaaX8x4WHMGtoYrCpJ6H1t1uhZgnU1riebYNZV1ui2Fs9fe5c4rVJP8fjeXN+CbuvR1h97z75sTvOPh79ndGX27o2uthotTCOfmBMc+UPo5yn7Amee1XavWJT2ygxgqP/3r9x7A/TnHNR8H5MnscuCDI368EnT94408Ea+mfYnNdUgaGfPyjGgPB1/fH9s9LF99Sfjq2RSR3/U7JP94OfMlJQ0K9f2EasixQKO7KDsOYHkoYnIm+LuGeXpHFcTZxxNM85BKl/DaRE99R7Clf66NWC8YS+FCHKjd524ymUUn15V3F031VH3OOFvpv6RSlpwNZPr1miIchFqvBJXSh3cI/CqWxkvtT/G1jW8WqUT97tnB+O8Dksotvo0w/dRBl2/zvFQiYVCrDvLRC5vNUvTyiMhvuu8E+8IloPDPr6WoohYY/Ag26ugwcg9mdfTQ0wv95fM6oN+ew0uri9R7I9vt8F2YvBBxAh1JcIITRWogQlO60WXEJeWOWY9b2d/dbkKJzrUkIfcb8fFKtH+wWR77EakZOz8NqpS7KGF8yFqJ04OAwfi+nZyC8PD1EAXeYW4wAPo8qEoiLjcac2vIdWbGdiQV4rkBxU41PsQrdNsPahwxtG6rHSNez5Oq+u2/sny8TiDRBt+W5pMt18c7cKWNMOsluYNj6Wdwp/AXhKPmlSSVctiX0q6t/8WtF2NmzJQbpDs70kV/kyTY3uOO57f83Lgqg7nuVnhnzvNm/vetu3ZX8BM0fVwLFk/htn4mmrQUr1AnRy52BQUCTqAZKCA0wGyfepFoG3NCDTtJIqjFmqzlVoiVVOKvHtrDSTui/7TQWSECcj13ZQn4oXalI/nuYpNwyTn+sW8fNzQCx2vNgxOH7WN8Ph4Li2l+xbD+j2X1AFjLP/rCW7yv/SaqtIoNKJHGznPPdowGlGlkVJ30f/g9iidOcGQzkAi8eUmxXd18KARYq5IvoQRGk2EcpjbaFIoVw/YyqpcQ5NbFUHI/Cqncibuliit+giv1lLJXQVmi3+LlsSydlhdVFIraAup6iGWVqoeeKO7vodXlnaHX1JTfNJGe9gGjgD8qz1BVh1BzWb8D26z2AgljY7S7SGw8vTZt4H6+rkvP5lSkNDT3zq6wRgoxK/5y2QTpwvknVU7ipP9L3Qt2PJgz1+KYB3FkPL2t97MvuZfVzLKqGJPASpYH+ocKw8uSx2Le1bkGlneujtPfRZGXZD38kv6M7rJxh0pBrSHLMRU4luB97GhcH6NasZa7kPQjmVkXNtf1hWnFwe3OEsQIOJLVt2QltZfZPdFHEmG/W8fhwWHZ0aCN/H9+M93lz+dX9peJwelbXo3zYC7nl3cqK8DaiKLTz3c4JutTOIEyJzMOFU+a3jayccD4WtW3yKkkqMgK23Uq0a9LFtVm2BLObHMbTW9Tqr5FH5EH58O5dmSvfrcxr1vWNtgmdQcIXRW6QxJnunSd8e6uIUBPgmgwdH3ujk4Zku4xBdH//ycphqLpaUG92c0xC7g9lAteVNa7x/Z+/DVxduLqx/PX5pIYNfrGXTOhAKHN77ULov9vdelB3af03Wn2SHr5dz+2Db6xJK6YpCRtgr+r431h2tzTeE+WiyqZAEh0pDq8ZrFeWAW9p/Rj10TjSDFtAu8M9rSu2APzmkPjxRVmrffXGRZYs5hW/xFCeOrljKoPuFazYB5zU1Xki7PtjS4Lws6vxRxvXeQELSzCtq+W4R2x1VlFHcILdu0WqlPTDyYPFxS59S+ddHbIktvi/gjqq/hqkwi+DQWkhKaqgL62WkJ8AsddKqxe/ZDZJCFF4tghWzDulPbM0bgbMMaX7uhUKwnyuU8qPrCZHl4q2m4T4JraquNz2tiGjqO1/AKLXtOnwJdPdjhg/Bzjt3IkGhtB54jiw9JO1dOmdO5W0MMMBRFUA4Tj6FfVZV2plalj1VfGh6Gh/XBeV3kumnnFP1RP33q+rU+to+f2t3/uDOYyvbz96RK51v1IR25yE7kZjtpc/6IjMmuR/dHiIlPm6Sf2wkJk7CxRiUCQii6+G9xBtteakcDZ1ozf6uJDVuLPsZWcbgqNwlX9u/nGduvbLkkgnk2R8XpBEpKKaP160rN+ZW92/u9XONO5383guG+cAjh7Hr98RHWbxVz2KsA+rJzV+OzUKEiNPuXMxGeQ/U72e3AdX7/Nu30ZCj22su2vjcRp26lkBKWTcJH+N7q29LAWHg7bBEDtYcPgrCIouvECnzcncEMi3JPoIetrg5WXaZydOtOJnZf1kY7JV1339ibD/EMoZj/lvxJ+1aItqdT9kKBrhzmTrtugKc9tTQMmL7xMQbb/7XAl4Yi8cbifAI4GkSkpmLDbeSktK0w/V2w1+vVCto7jEvSY3wI4t9Y7yN25I+TjMgUwjChs+yHcmqU9eCAzgYP4F9RQjDk9JY0WSSZ0gkQCY4Th2DZdyVy/4wyU8gLzNA2dzq571Tfv2vew/vT2xMoIcU8SrSXZrJKqCtHz9kjueLLx5967PO4YecifI1Kd2frL2vJiZMvNH7BXPlzWWqyE5s3J7QfS2fToH2jCo53Ch4bEzFWvYsgJW99KFsfdiEPhxCl/3lWEmHKrGhIfIQhsf3hE9V37LPFgjiZvZp+ucUTZMUya6jd70E9cep9LKkfQBK83VHFwSbtMpeYLrxoE1DnPgmxGKt2YNPhL1EPB96z9huvElrHn9X5vcuSqTbZHUUbH62dhd/HB2/h97eQj5Z8+kB9CAAB1UkV2AWreKkjHPItv3OIhxC3WGls/Yf17E+mdvjV+0uCW9smXTn4uDYKsmQBBmZeEA3Eap4jKH/hN9DAhFm0hfjjCgT8OrpN6O8O8WVME8tol3HaLxG0FJdpMHw6cMcA3CwraAupvXvvx1/191qyQoW+8KWmWTl4xUA435EwTcKOIFAw3+vaTK4sJfjel7ApttfyUPrB63in9kkLXcEOK+MdicVI9HP79lv3kVmurLZKd9rZA7QXL/XPoanszMRffbb/ID69F5DOXwLrYtXbs4n+hO/o0gYSZT1gMeCQGLZGS/yN9jtXtmQRaTrerJauSSYwP0wjaRktdoE1/AsRKAufbZ17di+8oO1MxeicqJgdqwt0Yc/zNloH9wSukTaYTX8bgeMupTvtjFvCRyRl93s4K8DfkcNxiJssjEAcLPDs5/eXLwDYjPPvVAjfmaUSPdfmX0bfZxyjJwukU8WbIxmAtS9EL9xK2UxUTdktyb+rVyQ5j/kzY7+nJUJK/f+dlvh4sdNGJd8foO7/DoWMfWd7beOL9+HL81evzz6cv2Sspi2o+UsOwTMX5+z3kfk5ahUrrT9GbbcJecruq+Oj30Vf4pQepJuMvw4RNoWeoLsbXfTh/OpD+9mfl3Z3uFjP1J45m+bKAUqft8petFGK1GaQxMOQ6HgYIoFdGMLzqQwI5vzBQv+sWqzB3b3HrwofUvOjOA4jVezYx8fy2VjJW4Ozp6IlAVOWxXBvas0Cx8hLdKKmPvsd6mY695KnUgAhi+t9tesD1ZGpLArgmkhzR5vaMe5KYtd8TJ3Wrmc82olkZqWYWVHNbNRzhoh8efD0Q91q03b1KmaNPRVmp4ZkTBAt5Lje4gt9aePQav4vMYuPeQ=="
engine.write_bytes(zlib.decompress(base64.b64decode(encoded_engine)))
print('Engine bytes:', engine.stat().st_size)
print('Engine SHA256:', hashlib.sha256(engine.read_bytes()).hexdigest())
shutil.copy2(engine, out / 'edge_fixedtest_engine.py')

In [ ]:
env = os.environ.copy()
env['EDGE_AUDIT_ROOT'] = str(work)
env['OPENBLAS_NUM_THREADS'] = '1'
env['OMP_NUM_THREADS'] = '1'
env['MKL_NUM_THREADS'] = '1'
env['NUMEXPR_NUM_THREADS'] = '1'

def invoke(args, filename):
    logfile = out / filename
    with logfile.open('w') as stream:
        completed = subprocess.run([sys.executable, str(engine), *args],
                                   stdout=stream, stderr=subprocess.STDOUT, env=env)
    excerpt = logfile.read_text(errors='replace')[-5500:]
    print(filename, 'exit:', completed.returncode, excerpt, flush=True)
    if completed.returncode:
        raise RuntimeError('Stage failed: ' + filename + ' (see full log in Kaggle Output)')

invoke(['--mode','prepare'], 'prepare.log')

In [ ]:
workers = min(2, torch.cuda.device_count()) if torch.cuda.is_available() else 1
workers = max(1, workers)
print('Launching', workers, 'worker(s)', flush=True)
processes = []
for wid in range(workers):
    handle = (out / f'worker_{wid}.log').open('w')
    proc = subprocess.Popen([sys.executable,str(engine),'--mode','worker',
                             '--worker',str(wid),'--workers',str(workers)],
                             env=env,stdout=handle,stderr=subprocess.STDOUT)
    processes.append((wid,proc,handle))
failed = []
for wid,proc,handle in processes:
    code = proc.wait();handle.close()
    log = (out/f'worker_{wid}.log').read_text(errors='replace')
    print(f'Worker {wid}: exit={code}\n{log[-5000:]}', flush=True)
    if code != 0: failed.append(wid)
if failed:
    raise RuntimeError(f'Worker failures: {failed}; inspect worker_*_failures.json and logs')
invoke(['--mode','aggregate','--workers',str(workers)],'aggregate.log')
archive=work/'EdgeIIoT_FixedTest_Results.zip'
print('RESULT ZIP:',archive, 'bytes:',archive.stat().st_size, flush=True)
from IPython.display import display, FileLink
display(FileLink(str(archive)))

In [ ]:
import pandas as pd
summary_file = out / 'summary_mean_std.csv'
status_file = out / 'run_status.json'
if status_file.exists():
    print(json.dumps(json.loads(status_file.read_text()),indent=2))
if summary_file.exists():
    result=pd.read_csv(summary_file)
    display(result[(result.policy=='none') & (result.feature_set=='expanded36') & (result.target_val_fpr==0.05)].round(4))